In [1]:
import pandas as pd
import os
import re
import nilearn
import numpy as np
import matplotlib.pyplot as plt
from nilearn.image import index_img, load_img, math_img
from nilearn.glm.first_level import FirstLevelModel
from nilearn.plotting import plot_stat_map, plot_stat_map, plot_design_matrix
import pandas as pd
from nilearn import datasets, image


In [49]:
#Masking for amygdala

atlas = datasets.fetch_atlas_harvard_oxford('sub-maxprob-thr25-1mm')
labels = atlas.labels
atlas_img = atlas.maps

print([i for i, name in enumerate(labels) if 'amygdala' in name.lower()])

amygdala_mask = math_img("np.isin(img, [10, 20])", img=atlas_img)

[get_dataset_dir] Dataset found in /home/bahriz1/nilearn_data/fsl

[10, 20]


In [90]:
task = "contact"
#subId = "sub-42"
contrasts = {'bra_vs_0': np.array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0.]),
            'fi_vs_0': np.array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0.]),
            'ru_vs_0': np.array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0.]),
            'so_vs_0': np.array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0.]), 
            'bra_vs_others': np.array([1., -1/3, -1/3, -1/3, 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0.]),
            'ru_vs_others':np.array([-1/3, -1/3, 1, -1/3, 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0.]), 
            'so_vs_others': np.array([-1/3, -1/3, -1/3, 1, 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0.]), 
            'fi_vs_others': np.array([-1/3, 1, -1/3, -1/3, 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0.]), 
            'black_vs_white': np.array([0.5, -0.5, -0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), 
            'white_vs_black': np.array([-0.5, 0.5, 0.5, -0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), 
            'others_vs_fi': np.array([1/3, -1, 1/3, 1/3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), 
            'warm_vs_cold': np.array([0.5, 0.5, -0.5, -0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),
            'cold_vs_warm': np.array([-0.5, -0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
            }

In [57]:
def log_file_analysis(task, subId, run):

    ###### input ######
    # task = "contact" (string "contact", "distance", "emotions" )
    #subId = "sub-01" (string sub-<two digit number>) 
    # run = 1 (int)

    

    # HRADCODED PATH
    folder_path = f"/scratch/nbe/prevent/Experiment1/Data_BIDS/derivatives/keras/{subId}/logs"

    log_file_name = f"virtual_contact_sce{run}.log"
    file_path = os.path.join(folder_path, log_file_name)

    
    expected_columns = ["Subject", "Trial", "Event Type", "Code", "Time", "TTime",
        "Uncertainty", "Duration", "Uncertainty", "ReqTime",
        "ReqDur", "Stim Type", "Pair Index"]
    
    # Regex pattern to match only valid Subject lines like S1-Se1, ..., S42-Se5
    valid_line = re.compile(r'^(s|sub)[-_]?([1-9]|[1-4][0-9])[-_]se[1-5][^\t]*\t', re.IGNORECASE)


    # Initialize list for rows
    rows = []
    columns = None
    correction_count = 0
    
    with open(file_path, "r") as f:
        for line in f:
            stripped = line.strip()
    
            # extract the header of the log file and check it with expected headers
            if stripped.startswith("Subject\t"):
                columns = stripped.split("\t")
                try:
                    if columns != expected_columns:
                        raise ValueError("Column mismatch")
                except ValueError as e:
                    print("❌ Error:", e)
                    print("File columns:     ", columns)
                    print("Expected columns: ", expected_columns)
                    raise
                continue
    
            # if the line is not a header, check that it is a data line and then extract rows from it
            if valid_line.match(line):
                parts = stripped.split("\t")
    
                # in case row has some missing values replace it with blank string and warn the user.
                if len(parts) < len(columns):
                    missing = len(columns) - len(parts)
                    parts += [""] * missing
                    correction_count += 1
                    
                    # Uncomment the following block to print details about each corrected line
                    # print(f"\n⚠️ Warning: Line {line} has {len(parts)} columns, expected {len(columns)}.")
                    # print(f"➤ Added {missing} empty string(s) to match column count.")
                    # print(f"Original line: {stripped}")
                    
                rows.append(parts)


    if correction_count > 0:
            print(f"⚠️ {correction_count} line(s) had missing columns and were auto-corrected with empty values.")



    df = pd.DataFrame(rows, columns=columns)
    
    # Convert "Time" column values to numeric. 
    df['Time'] = pd.to_numeric(df['Time'], errors='coerce') 
    
    
    # Filter for rows where 'Code' ends with '_vid' and ignore Ukraniane type
    df_vids = df[
        df['Code'].str.endswith('_vid', na=False) &
        ~df['Code'].str.startswith(('uk1_', 'uk2_'), na=False)
    ]
    
    # Build final dataset
    event_df = pd.DataFrame({
        "onset": df_vids["Time"]* 1e-4,
        "duration": 35,
        "trial_type": df_vids["Code"].str.replace(r'(1|2)_vid$', '', regex=True)
    })
    
    event_df = event_df.reset_index(drop = True)
    
    frame_times = df[df['Code']== "30"]['Time']* 1e-4
    frame_times = frame_times.to_numpy()

    ################# end of func ########################


    return event_df, frame_times

In [58]:
def nifti_finder(task, subId, run):

    ###### input ######
    # task = "contact" (string "contact", "distance", "emotions" )
    #subId = "sub-01" (string sub-<two digit number>) 
    # run = 1 (int)

    # HARDCODED_PATH
    
    nifti_folder_path = f"/scratch/nbe/prevent/Experiment1/Data_BIDS/derivatives/fmriprep/{subId}/func"
    #nifti_folder_path = f"/scratch/nbe/prevent/Baran/Codes/trimmed_nifti/{subId}"

    run_str = f"run-{run:02}"

    matched_nifti_path = None
    for fname in os.listdir(nifti_folder_path):
        if (
            "2009" in fname
            and task in fname
            and fname.endswith("bold.nii.gz")
            and run_str in fname
        ):
            matched_nifti_path = os.path.join(nifti_folder_path, fname)
            print(f"✅ Found file for {run_str}: {fname}")
            break 

    else:
        raise ValueError(f"❌ No file found for {run_str}")
        
        
    return matched_nifti_path

In [59]:
def confound_generator(task, subId, run, frame_times_size):

    ###### input ######
    # task = "contact" (string "contact", "distance", "emotions" )
    #subId = "sub-01" (string sub-<two digit number>) 
    # run = 1 (int)

    rows_to_trim = 0 
    
    matched_nifti_path = nifti_finder(task, subId, run)
    confounds, sample_mask = nilearn.interfaces.fmriprep.load_confounds(
        matched_nifti_path, strategy=('motion', 'wm_csf','global_signal'), motion='basic', wm_csf='basic', global_signal='basic')
    
    # if nii.gz data has more volumes in the end of the experiment, we drop them 
    if len(confounds) > frame_times_size:
        rows_to_drop = len(confounds) -  frame_times_size
        confounds = confounds.iloc[:frame_times_size] 
        print(f"⚠️ Dropped {rows_to_drop} extra row(s) from the bottom.")

    elif len(confounds) < frame_times_size:
        rows_to_trim = frame_times_size - len(confounds)
        print(f"⚠️ Need to trim {rows_to_trim} rows from frame_times")

        
    return confounds, rows_to_trim

In [60]:
def dm_generator(task, subId, run):
    event_df, frame_times = log_file_analysis(task, subId, run)
    confounds, flag26 = confound_generator(task, subId, run, len(frame_times))

    if flag26 != 0:
        frame_times = frame_times[:-flag26]
        
    
    design_matrix = nilearn.glm.first_level.make_first_level_design_matrix(frame_times, event_df, hrf_model = "spm", add_regs = confounds, drift_model='cosine')

    #plot_design_matrix(design_matrix)
        
    return design_matrix

In [61]:
def get_trimmed_nifti_paths(subId):
    sub_dir = f"/scratch/nbe/prevent/Baran/Codes/trimmed_nifti/{subId}"
    paths = []

    for run in range(1, 6):
        file_name = f"trimmed_bold_run{run}.nii.gz"
        file_path = os.path.join(sub_dir, file_name)
        if os.path.exists(file_path):
            paths.append(file_path)
        else:
            print(f"  {sub_id} run {run}: ❌ missing")

    return paths

In [68]:
def subject_con_map(subId, task):
    
    #dms = list of pandas df, len(runs per sub), each df is a design matrix for one run
    #paths = list of strings, len(runs per sub), each str is a path to a nifti image for one run
    
    
    dms = [dm_generator(task, subId, run) for run in range(1, 6)]
    trimmed_paths = get_trimmed_nifti_paths(subId) 
    
    first_level_model = FirstLevelModel(t_r=1.6,hrf_model='spm')
    
    # Fit model to all 5 runs
    first_level_model = first_level_model.fit(trimmed_paths, design_matrices= dms)

    #TODO: smoothing 4 mm

    # design_columns = first_level_model.design_matrices_[0].columns.tolist()
    # contrast_dict = make_contrasts(design_columns)


    run_contrasts(first_level_model, contrasts, out_dir=f"/scratch/nbe/prevent/Baran/Codes/beta_maps/{subId}")

    return None

    

In [86]:
def run_contrasts(first_level_model, contrasts, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for name, vec in contrasts.items():
        print(f"→ running contrast: {name}")
        z_map = first_level_model.compute_contrast(vec, output_type='z_score')

        #new_map, threshold = nilearn.glm.threshold_stats_img(z_map, alpha = 0.05, height_control = 'fdr')

        # Save z-map
        z_map_path = os.path.join(out_dir, f"{name}.nii.gz")
        z_map.to_filename(z_map_path)

        if z_map is None:
            print(f"❌ Contrast {name} produced None")
            continue

        
        # Plot z-map
        plot_stat_map(
            z_map,
            title=name,
            display_mode='ortho',
            threshold=3,
            output_file=os.path.join(out_dir, f"{name}.png")
        )


In [64]:
def make_contrasts(design_columns):

    conditions = ['bra', 'fi', 'ru', 'so']
    contrasts = {}

    # condition vs baseline
    for cond in conditions:

        contrast = np.array([1.0 if col == cond else 0.0 for col in design_columns])
        contrasts[f"{cond}_vs_0"] = contrast

    # condition vs others
    for cond in conditions:
        contrast = np.zeros(len(design_columns))
        others = [c for c in conditions if c != cond]
        for i, name in enumerate(design_columns):
            if name == cond:
                contrast[i] = 1.0
            elif name in others:
                contrast[i] = -1.0 / len(others)
        contrasts[f"{cond}_vs_others"] = contrast

    return contrasts

In [89]:
for sub in range(1,43):
    subId = f"sub-{sub:02}"
    subject_con_map(subId, task)



⚠️ 298 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-01: sub-01_task-contact_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 299 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-02: sub-01_task-contact_run-02_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 295 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-03: sub-01_task-contact_run-03_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 295 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-04: sub-01_task-contact_run-04_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 297 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-05: s

/appl/scibuilder-mamba/aalto-rhel9/prod/software/neuroimaging-env/2025.01/18c1437/lib/python3.11/site-packages/nilearn/glm/first_level/first_level.py:585: UserWarning: Mean values of 0 observed. The data have probably been centered.Scaling might not work as expected
  Y, _ = mean_scaling(Y, self.signal_scaling)


→ running contrast: fi_vs_others


/tmp/ipykernel_780749/2496362743.py:5: UserWarning: One contrast given, assuming it for all 5 runs
  z_map = first_level_model.compute_contrast(vec, output_type='z_score')


→ running contrast: black_vs_white
→ running contrast: white_vs_black
→ running contrast: others_vs_fi
→ running contrast: warm_vs_cold
→ running contrast: cold_vs_warm
→ running contrast: comp_vs_non
→ running contrast: non_vs_comp
⚠️ 296 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-01: sub-02_task-contact_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 20 extra row(s) from the bottom.
⚠️ 297 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-02: sub-02_task-contact_run-02_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 297 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-03: sub-02_task-contact_run-03_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 297 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for

...

→ running contrast: black_vs_white
→ running contrast: white_vs_black
→ running contrast: others_vs_fi
→ running contrast: warm_vs_cold
→ running contrast: cold_vs_warm
→ running contrast: comp_vs_non
→ running contrast: non_vs_comp
⚠️ 296 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-01: sub-42_task-contact_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 296 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-02: sub-42_task-contact_run-02_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ 296 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-03: sub-42_task-contact_run-03_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
⚠️ Dropped 1 extra row(s) from the bottom.
⚠️ 295 line(s) had missing columns and were auto-corrected with empty values.
✅ Found file for run-04: sub-42_task-contact_run-04_space-MN

/appl/scibuilder-mamba/aalto-rhel9/prod/software/neuroimaging-env/2025.01/18c1437/lib/python3.11/site-packages/nilearn/glm/first_level/first_level.py:585: UserWarning: Mean values of 0 observed. The data have probably been centered.Scaling might not work as expected
  Y, _ = mean_scaling(Y, self.signal_scaling)


→ running contrast: fi_vs_others


/tmp/ipykernel_780749/2496362743.py:5: UserWarning: One contrast given, assuming it for all 5 runs
  z_map = first_level_model.compute_contrast(vec, output_type='z_score')


→ running contrast: black_vs_white
→ running contrast: white_vs_black
→ running contrast: others_vs_fi
→ running contrast: warm_vs_cold
→ running contrast: cold_vs_warm
→ running contrast: comp_vs_non
→ running contrast: non_vs_comp


In [ ]:
#TODO visualization in free surfer one for each contrast
# 2nd level analysis
# 35s dusration
# cleaner and running version of /scratch/nbe/prevent/mathesis_folder_tammilehto/data_analysis_code/fmri-mvpa/src/mvpa_train_best_model
# save the weights of the classifier after training
#